# Visual-language temporal boundary alignment
Run from the top in a GPU Colab runtime. Add a `GITHUB_TOKEN` Colab secret with read access to the private repository.

In [ ]:
# Clone or update the private repository, then install dependencies.
from pathlib import Path
import base64, getpass, os, subprocess, sys

REPO = 'sanay-tralshawala/video-language-temporal-boundaries'
REPO_DIR = Path('/content/video-language-temporal-boundaries')
try:
    from google.colab import userdata
    token = userdata.get('GITHUB_TOKEN')
except Exception:
    token = None
if not token:
    token = getpass.getpass('GitHub token with read access to the private repository: ')
if not token:
    raise RuntimeError('GITHUB_TOKEN is required because this repository is private.')
credential = base64.b64encode(f'x-access-token:{token}'.encode()).decode()
git_auth = ['git', '-c', f'http.extraHeader=AUTHORIZATION: basic {credential}']
if (REPO_DIR / '.git').is_dir():
    result = subprocess.run(git_auth + ['-C', str(REPO_DIR), 'pull', '--ff-only'])
else:
    result = subprocess.run(git_auth + ['clone', '--depth', '1', f'https://github.com/{REPO}.git', str(REPO_DIR)])
token = credential = None
if result.returncode:
    raise RuntimeError('Repository clone/update failed; check GITHUB_TOKEN access.')
os.chdir(REPO_DIR)
requirements = REPO_DIR / 'requirements-colab.txt'
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(requirements)], check=True)
print('Repository ready:', REPO_DIR)

In [ ]:
# Mount persistent output storage and report the accelerator.
from google.colab import drive
drive.mount('/content/drive')
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available(): print('GPU:', torch.cuda.get_device_name(0))

## Choose the study mode
Keep `two_video_trial` for a quick check. Use `stratified_24` for 24 reproducibly sampled long videos with at least three from each domain and maximum subcategory coverage. The full mode cycles through official archives, extracts selected files, and deletes each ZIP.

In [ ]:
STUDY_MODE = 'two_video_trial'  # change to 'stratified_24' for the full study
OUTPUT_DIR = '/content/drive/MyDrive/video_boundary_results'
DATASET_DIR_24 = '/content/Video-MME-stratified-24'
SAMPLE_SIZE = 24
MIN_PER_DOMAIN = 3
RANDOM_SEED = 0
FRAME_INTERVAL = 2.0
SUBTITLE_WINDOW = 15.0

In [ ]:
# Prepare either the two-video trial or the balanced 24-video dataset.
from pathlib import Path

VIDEO_LIST = None
METADATA_CSV = None
if STUDY_MODE == 'two_video_trial':
    from download_trial_subset import download_two_video_trial
    DATASET_DIR = str(download_two_video_trial('/content/Video-MME-two-video-trial'))
elif STUDY_MODE == 'stratified_24':
    import pyarrow.parquet as pq
    from download_selected_subset import download_selected_subset
    from sample_videos import (build_pipeline_manifest, official_metadata_path,
                               sample_long_videos, write_selection)
    manifest_dir = Path(OUTPUT_DIR) / 'manifests'
    metadata = official_metadata_path('/content/video_mme_metadata')
    selected = sample_long_videos(pq.read_table(metadata).to_pylist(), SAMPLE_SIZE, RANDOM_SEED, MIN_PER_DOMAIN)
    selection_csv = write_selection(selected, manifest_dir / 'long_stratified_24.csv')
    DATASET_DIR = str(download_selected_subset(selection_csv, DATASET_DIR_24))
    METADATA_CSV = str(build_pipeline_manifest(
        selection_csv, DATASET_DIR, manifest_dir / 'pipeline_manifest.csv'))
    domain_counts = {}
    for row in selected: domain_counts[row['domain']] = domain_counts.get(row['domain'], 0) + 1
    print(f'Selected {len(selected)} long videos across', len(domain_counts), 'domains:', domain_counts)
    print('Covered subcategories:', len({(r['domain'], r['sub_category']) for r in selected}))
else:
    raise ValueError("STUDY_MODE must be 'two_video_trial' or 'stratified_24'")

In [ ]:
# Validate every selected video/subtitle pair before loading models.
from video_boundary_pipeline import discover_videos
pairs = discover_videos(DATASET_DIR, VIDEO_LIST, METADATA_CSV, max_videos=SAMPLE_SIZE)
expected = 2 if STUDY_MODE == 'two_video_trial' else SAMPLE_SIZE
if len(pairs) != expected:
    raise RuntimeError(f'Expected {expected} valid pairs, found {len(pairs)}')
print(f'Validated {len(pairs)} video/subtitle pairs')
for pair in pairs[:5]: print(' ', pair.video_id, pair.video_path.name, '+', pair.subtitle_path.name)

In [ ]:
# Run the cached analysis and save change plus stacked-similarity figures.
from video_boundary_pipeline import PipelineConfig, run_pipeline
config = PipelineConfig(dataset_dir=DATASET_DIR, output_dir=OUTPUT_DIR,
    frame_interval=FRAME_INTERVAL, subtitle_window=SUBTITLE_WINDOW,
    max_videos=expected)
results = run_pipeline(config, video_list=VIDEO_LIST, metadata_csv=METADATA_CSV)
display(results['alignment'])
display(results['stats'])